# Multiclass Car Classifier — Model Zoo

Math lives in `.py` files (`common/data.py`, `numpy_model/network.py`). This notebook only runs checks and plots.

Run from the repo root. 7 classes, input `64×64×3` flattened to 12288 features.

# Phase 0 — Data pipeline

Load images, resize to 64×64, flatten, scale pixels to [0, 1], and split 70/15/15 stratified. Shared by every model so they train on the same arrays.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from common.data import CLASS_NAMES, FLAT_DIM, load_dataset, to_one_hot
from numpy_model.network import (
    LAYER_DIMS,
    accuracy,
    backward,
    compute_loss,
    forward,
    init_params,
    train,
    update_params,
)

data = load_dataset()
for split in ("train", "val", "test"):
    X, y = data[f"X_{split}"], data[f"y_{split}"]
    print(f"{split:5}  X {X.shape}  y {y.shape}  range=[{X.min():.3f}, {X.max():.3f}]")
print("flat dim", FLAT_DIM, "classes", CLASS_NAMES)

# Phase 1 — Forward pass

Random weights, one batch through `12288 → 128 → 64 → 7` (ReLU, ReLU, softmax). Check that the output is `(m, 7)` and each row sums to 1.

In [ ]:
print("LAYER_DIMS", LAYER_DIMS)
Xb = data["X_train"][:32]
params = init_params()
for name, arr in params.items():
    print(f"{name:4} {arr.shape}")

A3, cache = forward(Xb, params)
print("A3", A3.shape, "row sums", A3.sum(axis=1)[:4], "...")
print("range", A3.min(), A3.max(), "cache", sorted(cache.keys()))

# Phase 2 — Backprop on a tiny subset

Cross-entropy loss, backward pass, and vanilla GD on 50 images. If loss does not fall here, the full dataset will not help.

In [ ]:
X_tiny = data["X_train"][:50]
Y_tiny = to_one_hot(data["y_train"][:50])
params = init_params()
losses = []
for step in range(80):
    A3, cache = forward(X_tiny, params)
    loss = compute_loss(A3, Y_tiny)
    grads = backward(A3, Y_tiny, cache, params)
    params = update_params(params, grads, learning_rate=0.1)
    losses.append(float(loss))
    if step % 20 == 0 or step == 79:
        print(f"step {step:3d}  loss {loss:.4f}")

plt.plot(losses)
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Phase 2 — 50 images")
plt.show()

# Phase 3 — Full training and diagnosis

Mini-batch GD on the full train set. Plot train vs val loss, then accuracy. Diagnosis is in 3.3.

In [ ]:
Y_train = to_one_hot(data["y_train"])
Y_val = to_one_hot(data["y_val"])

params, history = train(
    data["X_train"], Y_train,
    data["X_val"], Y_val,
    epochs=20, batch_size=64, learning_rate=0.5,
)

plt.plot(history["train_loss"], label="train")
plt.plot(history["val_loss"], label="val")
plt.axhline(np.log(7), linestyle="--", color="gray", label="chance (~1.95)")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.title("Phase 3 — train vs val")
plt.show()

A_train, _ = forward(data["X_train"], params)
A_val, _ = forward(data["X_val"], params)
print(f"final train loss {history['train_loss'][-1]:.4f}  val {history['val_loss'][-1]:.4f}")
print(f"train acc {accuracy(A_train, data['y_train']):.3f}  val acc {accuracy(A_val, data['y_val']):.3f}")

## 3.3 Diagnosis

This is **high bias**, not high variance. Train and val loss both sit at ~1.94 (chance is 1.95) with almost no gap, and both accuracies are 16.9%. That is not a memorizer (those have low train error and high val error). The net never fits the training set.

16.9% is also the share of the majority class (Convertible), not a fair 7-way guess (~14.3%). After epoch 1 the loss stops moving: the model is stuck predicting like a constant/majority classifier.

**Next:** raise the learning rate (or train longer) before changing the architecture — this may still be an optimizer problem. Regularization would be the wrong lever (that is for variance). If train loss still cannot leave ~1.94, then the flattened 64×64 pixels are the ceiling: neighboring pixels are unrelated features, and a bigger net or a CNN would be the real fix.